In [ ]:
import numpy as np
import pandas as pd
import os
import s3fs
import sys
import cartiflette #Attention, cartiflette n'est pas disponible par défaut sur Onyxia
sys.path.append('../outils')

import fonctions_nomenclature as fnom
import fonctions_preparation_table as fprep
%load_ext autoreload
%autoreload 2

np.random.seed(123)

# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})
BUCKET = "mathildedonnenfeld/diffusion/bistrots_et_bulletins"


FILE_KEY_S3 = "table_analyse_muni.csv"
FILE_PATH_S3 = BUCKET + "/" + FILE_KEY_S3

with fs.open(FILE_PATH_S3, mode="rb") as file_in:
    df = pd.read_csv(file_in, sep=",")

df.head()   

In [ ]:
df.columns

In [ ]:
# Modélisation simple, paramètres fixés

# Il va dans un seul bistrot par ville
nb_bistrots_par_ville = 1

# Dans 1000 bistrots au total
nb_bistrots_total = 1000

# On sélectionne donc les 1000 premières villes
nombre_villes = nb_bistrots_total/nb_bistrots_par_ville

# Paramètre de la stratégie de classement selon le nombre de bistrots 
croissant = True

# On crée la sous-table qui contient ces villes 
bistrots_visites = df.sort_values("nb_etablissements", ascending=croissant).head(int(nombre_villes)) #Arrondi à l'inférieur

# On calcule la proportion du nombre de bistrots visités dans la ville par rapport au nombre de bistrots totaux de la ville
bistrots_visites['part_des_etablissements_visites'] = nb_bistrots_par_ville/bistrots_visites['nb_etablissements']

# On applique cette proportion aux abstentionnistes de la ville
bistrots_visites['voix_pour_attal'] = bistrots_visites['part_des_etablissements_visites']*bistrots_visites['Abstentions']

# On calcule le nombre de voix obtenues par la liste gagnante
bistrots_visites['voix_LG'] = bistrots_visites['part_votes_liste_gagnante']*bistrots_visites['Exprimés']

# Indicatrice de victoire 
bistrots_visites['victoire_attal'] =  bistrots_visites['voix_pour_attal'] > bistrots_visites['voix_LG']

codes_communes_gagnees = bistrots_visites[bistrots_visites['victoire_attal']==True].CODGEO

decompte_communes = (bistrots_visites['victoire_attal'] == True).sum()
part_communes = decompte_communes / bistrots_visites.shape[0]

print(f"Gabriel Attal a emporté {decompte_communes} communes grâce à ses visites, "
      f"soit {part_communes:.1%} des communes visitées ! Bravo !")


In [ ]:
import matplotlib.pyplot as plt
from cartiflette import carti_download   # versions anciennes : from cartiflette.client import carti_download

# Fond de carte : communes de France, DROM rapprochés
communes = carti_download(
    values=["France"],
    crs=4326,
    borders="COMMUNE_ARRONDISSEMENT",
    vectorfile_format="parquet",   
    simplification=50,
    filter_by="FRANCE_ENTIERE_DROM_RAPPROCHES",
    source="EXPRESS-COG-CARTO-TERRITOIRE",
    year=2025,
)

# Départements, pour avoir des repères sur la carte
departements = carti_download(
    values=["France"],
    crs=4326,
    borders="DEPARTEMENT",
    vectorfile_format="parquet",
    simplification=50,
    filter_by="FRANCE_ENTIERE_DROM_RAPPROCHES",
    source="EXPRESS-COG-CARTO-TERRITOIRE",
    year=2025,
)

# Ta liste de codes communes (en texte !)
codes = [str(c).zfill(5) for c in codes_communes_gagnees]
communes["selection"] = communes["INSEE_COG"].isin(codes)

fig, ax = plt.subplots(figsize=(9, 9))
communes.plot(ax=ax, color="#e8e8e8", linewidth=0)
communes[communes["selection"]].plot(ax=ax, color="#c0392b", linewidth=0)
departements.boundary.plot(ax=ax, color="white", linewidth=0.4)
ax.set_axis_off()
ax.set_title(f"{communes['selection'].sum()} communes gagnées par Gabriel Attal, félicitations !")
plt.show()

In [ ]:
# Modélisation plus complexe (à transformer en version interactive ?)

#Paramètres à fixer :

# Nb de bistrots par ville

#Nb de bistrots complet

# Stratégie aggressive ou passive (taux appliqué aux abstentionnistes ou volé à la LG)

# Sélection des bistrots (par ordre croissant ou décroissant de nb de bistrots, de revenu médian, de taux de pauvreté, de % de la LG)
